# DEM and optical preprocessing walkthrough

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from rs_toolkit import io, terrain, optical, normalization, tiling
from rs_toolkit.visualization import show

## 1. Load (replace paths with your GeoTIFFs)

In [ ]:
dem_arr, dem_prof = io.read_raster('dem.tif', [1])
opt_arr, opt_prof = io.read_raster('optical.tif')
res = (abs(dem_prof['transform'].e), abs(dem_prof['transform'].a))

## 2. Derive terrain layers (DEM) and grayscale/edges (optical)

In [ ]:
dem = terrain.fill_voids(dem_arr[0])
slope, hs = terrain.slope(dem, res), terrain.hillshade(dem, res)
gray = optical.robust_stretch(optical.to_gray(opt_arr))
edges = optical.gradient_magnitude(gray[0])

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
show(hs[None], title='Hillshade', ax=ax[0]); show(gray, title='Optical', ax=ax[1]); show(edges[None], title='Edges', ax=ax[2])

## 3. Normalize, tile, split spatially
DEM and optical must be on the same grid first (`preprocessing.reproject_raster`).

In [ ]:
stack = normalization.zscore(np.stack([slope, hs, gray[0], edges]))
patches, xy = tiling.extract_patches(stack, 256, overlap=32)
tr, va, te = tiling.spatial_split(xy, 256)
len(tr), len(va), len(te)